In [ ]:
################################################
# New style 
################################################
import sys
import os

workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

from myPythonTools.Utils import utils as uti
from myPythonTools.Utils import numerical_utils as nuti
from myPythonTools.Utils import AveragingUtils as Av
from myPythonTools.Utils import validation_data as Val
from myPythonTools.Utils import PlotUtil as Pu
from myPythonTools.Plotting import LatLonMaps as LL 
from myPythonTools.Plotting import utils as puti
from myPythonTools.CASutils import filter_utils as fu 

from PyRegridding.Utils import GridUtils as GrU
from PyRegridding.Utils import MakePressures as MkP
from PyRegridding.Drivers import RegridField as RgF


# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.tri as tri

# Cartopy for pretty maps
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Some other useful packages 
import importlib
import copy
import time
import cftime

importlib.reload( uti )
importlib.reload( puti )
importlib.reload( nuti )
importlib.reload( Pu )
importlib.reload(Av)
importlib.reload(Val)
#importlib.reload(vAB)
importlib.reload(MkP)
importlib.reload(RgF)


In [ ]:
from myPythonTools.YAML import get_exp_list as Xl
importlib.reload(Xl)


In [ ]:
print( sys.path )
############################
# Initialize dict for regridding
############################
RgObs={}


In [ ]:
### Make a list of files
import yaml
#from box import Box


with open('../YAML/configure_plotting.yaml', 'r') as file:
    cfgPlot = yaml.safe_load(file)


######################################################
season =  'jja' #'jja' #cfgPlot['zonal_mean']['season']
valData  = cfgPlot['zonal_mean']['validation']['data']
valYears = cfgPlot['zonal_mean']['validation']['years']
print( f"Season= {season}" )
print( valData , valYears )

fld='T' #'QCP' 'U' #'QRS' #'U'
hsPat = 'cam.h0a' # assume fld is in same h-file for all exps. Not great.


regrid_val_x_model = False # True
all_x_fv1x1_zERA = False #True #True
use_desc_in_title = False #True
use_name_in_title = True #True
use_pmid_for_z =False #False
plot_jet_axis_range = False #True
use_ymdPat_from_yaml = True

# if you want to restrict years manually,
# do it here with override_ymdPat=True
#--------------------------------------
override_ymdPat = True
if (override_ymdPat==True):
    use_ymdPat_from_yaml = False
    ymdPat_spec=[1985,1985] # [1985,1987]


In [ ]:

#Xl.Fig1Paper()  #JointGWTune()  #Utility() 

#exp_file,xl= Xl.Ctl_c6477() #Xl.Fig1Paper()  #JointGWTune()  #Utility()
#exp_file,xl= Xl.KeyFig_Paper() 
#exp_file,xl= Xl.RadTest() 

#exp_file,xl= Xl.DevelControlRuns() 
#exp_file,xl= Xl.Official() 


#exp_file,xl= Xl.Fig1Paper() 

#exp_file,xl= Xl.Fig_for_NSC2026()

#exp_file,xl= Xl.fGW_tests() 


#exp_file,xl= Xl.tausrfRuns() 


#exp_file,xl= Xl.ModHeat() #Xl.ControlRuns() 
#exp_file,xl= Xl.JointGWTune()  

#exp_file='ChemWACCMRuns.yaml'
#xl=['fmt_cam','fmt_waccm','control_c64116']
#xl=['fmt_waccm',]

"""
exp_file='ControlRuns.yaml'
xl=['CTL_c64116','CTL_c64144']
exp_file='OfficialRuns.yaml'
xl=['B1850_x156','B1850_x271','B1850_x297',]
"""


#exp_file='fGW_tests.yaml'
#xl=['CTL_c64173','LatMM_LatFr_c64173','Lat3Fr_c64173','NoFr_c64173',]  #'X01_RdFGW_latMM_c64144',]
#xl=['CTL_c64173','NewMMsrc_c64173','NewMMsrc_NoFr_c64173','NewMMsrc_c64173-x01','NoFr_c64173',]  #'X01_RdFGW_latMM_c64144',]
#xl=['CTL_c64173','NewMMsrc_c64173','NewMMsrc_NoFr_c64173','NewMMsrc_c64173-x01','NoFr_c64173',]  #'X01_RdFGW_latMM_c64144',]

exp_file='fGW_tests.yaml'
#xl=['CTL_c64173','NewMMsrc_c64173','NewMMsrc_NoFr_c64173','NewMMsrc_c64173-x01','NewMMsrc_c64173-x02','NoFr_c64173',]  #'X01_RdFGW_latMM_c64144',]
#xl=['CTL_c64173','NewMMsrc_c64173-x01','NewMMsrc_c64173-x02','NewMMsrc_c64173-x03','NewMMsrc_c64173-x04',]   #'NoFr_c64173',]  #'X01_RdFGW_latMM_c64144',]




#xl=['CTL_c64173','NewMMsrc_c64173-x09','NewMMsrc_c64173-x11','CLUBBmod_c64173-x03',]   #'NoFr_c64173',]  #'X01_RdFGW_latMM_c64144',]
#xl=['CTL_c64173','DMF_c64173',]   #'NoFr_c64173',]  #'X01_RdFGW_latMM_c64144',]
#xl=['CTL_c64173','NewMMsrc_c64173-x04','NewMMsrc_c64173-x06','NewMMsrc_c64173-x07','NewMMsrc_c64173-x08','NewMMsrc_c64173-x09','NewMMsrc_c64173-x11','CLUBBmod_c64173-x03',]   #'NoFr_c64173',]  #'X01_RdFGW_latMM_c64144',]

#xl=['CTL_c64173','NewMMsrc_c64173-x11','TST01_aphysPR448','TST01_aphysPR448',]
xl=['CTL_c64173','TST01_aphysPR448','TST02_aphysPR448','TST04_aphysPR448','TST01_Xsgh_aphysPR448',]
xl=['CTL_c64173','TST01_aphysPR448','TST01_Xsgh_aphysPR448','TST01_PBLfix_aphysPR448',]
xl=['CTL_c64173','TST01_aphysPR448','TST01_Xsgh_aphysPR448','TST01_PBLfix_aphysPR448','TST01_Vortfix_PBLfix_aphysPR448','TST01_PBLfix_Xsgh200_aphysPR448',]
xl=['TST01_aphysPR448','TST01_Xsgh_aphysPR448','TST01_PBLfix_Xsgh200_aphysPR448','RV03_aphysPR448',]
xl=['CTL_c64173','TST01_aphysPR448','TST01_Xsgh_aphysPR448','TST04_Adpht01_aphysPR448','RV03_aphysPR448',]
xl=['CTL_c64173','RV03_aphysPR448','RV03_x02_aphysPR448','RV03_x03_aphysPR448',]
xl=['CTL_c64173','RV03_x03_aphysPR448','RV03_x03_dbg_aphysPR448','RV03_x03_radfix_aphysPR448',]



#xl=['CTL_c64173','NewMMsrc_c64173-x01','NewMMsrc_c64173-x02','NewMMsrc_c64173-x03','NewMMsrc_c64173-x04','NewMMsrc_c64173-x05',]   #'NoFr_c64173',]  #'X01_RdFGW_latMM_c64144',]
#xl=['CTL_c64173','LatMM_c64173','LatMM_45_c64173','NewMMsrc_c64173-x02','NewMMsrc_c64173-x03','NewMMsrc_c64173-x04',]   #'NoFr_c64173',]  #'X01_RdFGW_latMM_c64144',]
#xl=['CMIP6_WACCM','CTL_c64173','LatMM_c64173','LatMM_45_c64173','NewMMsrc_c64173-x02','NewMMsrc_c64173-x04',]   #'NoFr_c64173',]  #'X01_RdFGW_latMM_c64144',]



#xl=['NewMMsrc_c64173-x05','NewMMsrc_c64173-x05-bugtest-MM',]   #'NoFr_c64173',]  #'X01_RdFGW_latMM_c64144',]

#xl=['CTL_c64173','LatMM_c64173','LatMM_LatFr_c64173','LatMM_Lat2Fr_c64173','NoFGW_latMM_c64144',]  #'X01_RdFGW_latMM_c64144',]


#xl=['CTL_c64173','LatMM_c64173','LatMM_LatFr_c64173','NoFGW_c64144','NoMMGW_c64144','NoFGW_latMM_c64144',]  #'X01_RdFGW_latMM_c64144',]



exp_file=f'../YAML/{exp_file}'
with open( exp_file, 'r') as file:
    cfg = yaml.safe_load(file)

Al =[]
for x in xl:
    exp, subd, Src, Hkey, Dst, useri = cfg[x]['name'] , cfg[x]['subdir'] , cfg[x]['SrcGrid'] , cfg[x]['Hkey'] , cfg[x]['DstGrid'] , cfg[x]['user'] 

    if (use_ymdPat_from_yaml==True):
        ymdPat=cfg[x]['ymdPat']
        if ( 'ymdPat_is_range' in cfg[x] ):
            ymdPat_is_range = cfg[x]['ymdPat_is_range']
        else:
            ymdPat_is_range = False
    elif (override_ymdPat==True):
        ymdPat=ymdPat_spec
        ymdPat_is_range = True
    else:
        ymdPat = '*' # ['1985-*', '1986-*']
        ymdPat_is_range = False


    
    print( exp, subd, Src, Hkey, Dst, useri )
    print( ymdPat  )
    
    A = uti.MakeDict4Exp( exp=exp , user=useri, subd=subd , 
                        hsPat=hsPat , ymdPat=ymdPat , 
                        ymdPat_is_range =ymdPat_is_range , 
                        cmip_fld=fld ,
                        verbose=True , open_dataset=True )
    
    #A = uti.MakeDict4Exp( exp=exp , user=useri, subd=subd , 
    #                    hsPat=hsPat , ymdPat=ymdPat, cmip_fld=fld,
    #                    verbose=True, open_dataset=True )
    
    A['alias']=x
    A['Src']=Src
    if ('SrcVgrid' in cfg[x]):
        A['SrcVgrid']=cfg[x]['SrcVgrid']
        print( f"SrcVgrid for {x} = {A['SrcVgrid']} ")
    else:
        print( f'No SrcVgrid in {x}' )
    A['Hkey']=Hkey
    if 'gw_parameters' in cfg[x]:
        A['gw_parameters'] = cfg[x]['gw_parameters']
    if 'desc' in cfg[x]:
        A['desc'] = cfg[x]['desc']
    Al.append( A )
    print( f'Vertcal levs - {A.X.lev.shape}' )
    print( f'{A.exp} Years {len(A.X.time) /12. } \n')


In [ ]:



for A in Al:
    print( 'PINT' in list(A.X))

In [ ]:
########################################
# Make seasonal means
########################################

for A in Al:
    print( f'{A.exp} Years {len(A.X.time) /12. }')
    
    AAa_s,years_a,months_a = Av.Seasonal( ds=A.X, season=season , fld=fld,return_time=True)
    A['AA_s']=AAa_s
    A['years']=years_a
    A['months']=months_a

    if (use_pmid_for_z==True):
        PSa_s = Av.Seasonal( ds=A.X, season=season , fld='PS' )
        A['PS_s']=PSa_s
        PMIDa_s = Av.Seasonal( ds=A.X, season=season , fld='PMID' )
        A['PMID_s']=PMIDa_s
        PINTa_s = Av.Seasonal( ds=A.X, season=season , fld='PINT' )
        A['PINT_s']=PINTa_s



In [ ]:

########################################
# Make zonal means
########################################
method='C2'

for A in Al:
    Hkey = A['Hkey']
    Src  = A['Src']
    AAa_s = A['AA_s']
    if ('PS_s' in A):
        PSa_s = A['PS_s']
    if (use_pmid_for_z==True):
        PMIDa_s = A['PMID_s']
        PINTa_s = A['PINT_s']
    
    if (Hkey == 'yx' ):
        ########################
        # Option 1 from lat-lon regirrdded data
        ########################

        lon_a=A.X.lon.values
        lat_a=A.X.lat.values
        lev_a=A.X.lev.values
        ilev_a=A.X.ilev.values
        zlev_a=-7.0*np.log( lev_a/1_000. )
        zilev_a=-7.0*np.log( ilev_a/1_000. )

        AAa_sz=np.average( AAa_s , axis=2 )
        if (use_pmid_for_z==True):
            PMIDa_sz = np.average( PMIDa_s , axis=2 )
            zlev_a=-7.0*np.log( PMIDa_sz / 100_000. )
            PINTa_sz = np.average( PINTa_s , axis=2 )
            zilev_a=-7.0*np.log( PINTa_sz / 100_000. )
   
    elif (Hkey=='c'):
        ########################
        # Option 2 from unstructured SE data
        ########################
        
        Dst = 'fv1x1'
        DstInfo = GrU.gridInfo(Dst) #,Vgrid=DstVgrid)
        lat_a,lon_a = GrU.latlon( scrip= DstInfo['scrip'], Hkey=DstInfo['Hkey'] )
        lev_a=A.X.lev.values
        ilev_a=A.X.ilev.values
        zlev_a=-7.0*np.log( lev_a/1_000. )
        zilev_a=-7.0*np.log( ilev_a/1_000. )
    
        RegridObj = GrU.regrid_object_lib(RgOb=RgObs, src=Src, dst=Dst)
        AAa_s_xy=RgF.Horz(xfld_Src=AAa_s , Src=Src, Dst=Dst , RegridObj_In= RegridObj )

        # Zonal Av
        AAa_sz=np.average( AAa_s_xy , axis=2 )

        if (use_pmid_for_z==True):
            print( Src , Dst )
            print( 'PMIDa_s', np.shape(PMIDa_s) )
            PMIDa_s_xy=RgF.Horz(xfld_Src=PMIDa_s , Src=Src, Dst=Dst , RegridObj_In= RegridObj )
            print( 'PMIDa_s_xy', np.shape(PMIDa_s_xy) )
            PMIDa_sz = np.average( PMIDa_s_xy , axis=2 )
            zlev_a=-7.0*np.log( PMIDa_sz / 100_000. )
            print( 'PMIDa_sz',np.shape(PMIDa_sz) )
            print( 'zlev_a',np.shape(zlev_a) )

            PINTa_s_xy=RgF.Horz(xfld_Src=PINTa_s , Src=Src, Dst=Dst , RegridObj_In= RegridObj )
            PINTa_sz = np.average( PINTa_s_xy , axis=2 )
            zilev_a=-7.0*np.log( PINTa_sz / 100_000. )

    A['AA_sz'] = AAa_sz
    A['lonx'] = lon_a
    A['latx'] = lat_a
    A['zlev'] = zlev_a
    A['zilev'] = zilev_a
    A['plev'] = lev_a
    A['DateRange'] = f"{(A.X.time[0].values.item() ).strftime('%Y-%m-%d')}_{(A.X.time[-1].values.item() ).strftime('%Y-%m-%d')}"

In [ ]:
####
# Validation
####

valYears='*'
Aval = Val.data(fld=fld,season=season,zlev=True , Years=valYears ) #, Years='1996' ) #, Years='2000' )

if (Aval['data_exists']==True):
    print( f'Found validation data for {fld}' )
    print( " shape of Val data ",Aval['aa'].shape )

    AAv_s , zlev_v,lat_v,lon_v, years_Uv, Src, SrcVgrid = Aval['aa'], Aval['lev'], Aval['lat'], Aval['lon'], Aval['years'], Aval['data_Hgrid'], Aval['data_Vgrid']
    
    AAv_sz=np.average( AAv_s , axis=2 )
    A['AAv_sz']=AAv_sz


    if (use_pmid_for_z==True):

        PSval = Val.data(fld='PS',season=season,zlev=True , Years=valYears )

        PMIDv_s_xy, PINTv_s_xy , DELPv_s_xy = MkP.Pressure ( am = PSval['hyam' ] , bm = PSval['hybm' ] , ai = PSval['hyai' ] , bi = PSval['hybi' ] , 
                      ps = PSval['aa'] , p_00=100_000., Gridkey='zyx' )

        print( 'PMIDv_s_xy', np.shape(PMIDv_s_xy) )
        PMIDv_sz = np.average( PMIDv_s_xy , axis=2 )
        zlev_v=-7.0*np.log( PMIDv_sz / 100_000. )    
        
    Aval['AA_sz']=AAv_sz
    Aval['lon'] = lon_v
    Aval['lat'] = lat_v
    Aval['zlev'] = zlev_v
    
else:
    print( f'No validation data for {fld}' )



In [ ]:
print(Aval['data_exists'])
#print( PSval['aa'].shape )

In [ ]:
#print(np.shape(Al[1]['PMID_s']))
from myPythonTools.Zany import jetaxis as jax
importlib.reload( jax )

#plot_jet_axis_range=True
if ( fld=='U' and season=='jja' and plot_jet_axis_range==True):
    
    for A in Al:
        jaxes,mean,min,max,std,nyears = jax.make_axes_mean_range( X=A.X, Src=A.Src )
        
        A['jaxes']=jaxes
        A['jax_mean']=mean
        A['jax_min']=min
        A['jax_max']=max
        A['jax_std']=std
        A['jax_nyears']=nyears

        print( f'Years in {A.exp} = {nyears}' )
    
    Xval =xr.open_mfdataset( '/glade/campaign/cgd/amp/juliob/ERA5/fv1x1_monthly/ZY/ERA5.fv1x1_zERA_zonal.time*.nc' )
    
    uval=Xval.U.values
    print( uval.shape )
    
    nt = np.shape( uval )[0]
    nyears_val = nt // 12
    
    print( nyears_val )
        
    iseason = np.array( [6,7,8] ) - 1
    
    uval_s = np.stack([
        np.average(uval[y*12 + iseason , :, :], axis=0)
        for y in np.arange(nyears_val)
    ])
    
    
    
    jaxes_val=jax.lat_z( u=uval_s, lat=lat_a )
    
    mean_val = np.mean(jaxes_val, axis=0)
    min_val = np.min(jaxes_val, axis=0)
    max_val = np.max(jaxes_val, axis=0)
    std_val = np.std(jaxes_val, axis=0)
    
    Aval['jaxes']=jaxes_val
    Aval['jax_mean']=mean_val
    Aval['jax_min']=min_val
    Aval['jax_max']=max_val
    Aval['jax_std']=std_val
    Aval['jax_nyears']=nyears_val


In [ ]:
#print(np.shape(Al[1]['PMID_s']))
from myPythonTools.Zany import jetaxis as jax
importlib.reload( jax )

#plot_jet_axis_range=True
if ( fld=='U' and season=='jja' and plot_jet_axis_range==True):
    
    for A in Al[1:3]:
        jaxes,mean,min,max,std,nyears = jax.make_axes_mean_range( X=A.X, Src=A.Src , season=[1,2,3] )
        
        A['jaxes']=jaxes
        A['jax_mean']=mean
        A['jax_min']=min
        A['jax_max']=max
        A['jax_std']=std
        A['jax_nyears']=nyears

        print( f'Years in {A.exp} = {nyears}' )
    

In [ ]:
print( np.shape( Al[0].AA_sz ) )

plt.plot( 86_400. * Al[0].AA_sz[:,90] , zlev_a )
plt.plot( 86_400. * Al[1].AA_sz[:,90] , zlev_a )
plt.ylim( 0, 50 )
plt.xlim( -.5,.5 )


In [ ]:
use_desc_in_title = True

#nxplo,nyplo=4,1
#fig = plt.figure(figsize=(24, 5))

numx=len( Al )
plot_bias_vs_obs = True #True
if (plot_bias_vs_obs == True):
    nxplo,nyplo=numx+1,1
else:
    nxplo,nyplo=numx+1,1

latlims = (-90.,90. )
#latlims = (-90.,0. )
#latlims = (0.,90. )
plotTop= 85 #85 #85 #40 # 15 #10 #85


fig = plt.figure(figsize=(8*nxplo, 9))
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16

ulev=np.linspace( -60,140,num=21) 
ulevx = np.append(ulev, [160,180,200])

ulev=np.linspace( -0.05,0.05,num=41) 
ulevx=ulev


####################################
# Use a centarlized functiin to define 
# intervals 
####################################
#def contour_intervals( fld, **kwargs ): return flev,dlev,scale

#dlev=np.linspace( -20,20,num=21) 

ulev,dlev,scale = puti.contour_intervals( fld, zonal_mean=1 )

#ulev,dlev,scale = np.linspace( -1.,1., num=51 ) , np.linspace( -.01,.01, num=51 ) , 86_400.


#ulev=np.linspace(-450,450,num=51)
#ulev=np.linspace(-.03,.03,num=21)
ulevx=ulev
print(dlev)
cmap='bwr_r' #'gist_ncar'
cmap='gist_ncar'
titlesize=14 #24 #18

BoxParm =['alpha_gw_movmtn', 'effgw_rdg_resid','use_gw_rdg_resid']
BoxParm =['use_gw_convect_dp','use_gw_movmtn_pbl','use_gw_front','use_gw_rdg_resid','use_gw_rdg_beta','use_gw_oro']
BoxParm =['use_gw_movmtn_pbl','use_gw_rdg_resid','movmtn_source','alpha_gw_movmtn', 'effgw_movmtn_pbl','effgw_rdg_resid',]
BoxParm =['movmtn_source','alpha_gw_movmtn', 'effgw_movmtn_pbl','taubgnd','effgw_cm','effgw_rdg_beta','effgw_rdg_resid',]
#BoxParm =['use_gw_movmtn_pbl','movmtn_source','alpha_gw_movmtn', 'effgw_movmtn_pbl',]  #'effgw_rdg_resid',]


titles=[ r'1$^\circ$ control','14km simulation',r'1$^\circ$ w/ additional GW source','yy',]
titles=[ r'1$^\circ$ control',r'1$^\circ$ w/out GW parameterization','yy',]
ix=0

for A in Al:
    
    AAa_sz = scale * A['AA_sz'] 
    if (Aval['data_exists']==True) and ((regrid_val_x_model == True) or (all_x_fv1x1_zERA==True)) and (plot_bias_vs_obs==True) :
        AAa_sz = AAa_sz - A['AAv_sz']
        mlev=dlev
        cmap='bwr' #_r'
        #plotTop=65 #40 # 15 #10 #85
    else:
        mlev=ulev
        cmap='gist_ncar'
        #plotTop=85 #85 #40 # 15 #10 #85
        
    
    lon_a = A['lonx']
    lat_a = A['latx']
    zlev_a = A['zlev']
    years_a = A['years']
    if 'gw_parameters' in A:
        parameters=A['gw_parameters']
    else:
        parameters=None

    if (use_pmid_for_z==True):
        Zlev_a = zlev_a
        # Create 2D arrays
        lat_a, Lev_a = np.meshgrid(lat_a, A.X.lev.values )
    else:
        # Create 2D arrays
        Lat_a, Zlev_a = np.meshgrid(lat_a, zlev_a)
    
    # Adding the hatching for the sponge layer
    sponge_layer = np.zeros_like(AAa_sz)
    sponge_layer[(Zlev_a >= 62) & (Zlev_a <= 85)] = 1

    if ('desc' in A) and (use_desc_in_title == True):
        title = f"{A.exp}: \n {A.desc} \n {season.upper()} {years_a[0]}-{years_a[-1]}"
    elif ('exp' in A) and (use_name_in_title == True):
        title = f"{A.exp} \n {season.upper()} {years_a[0]}-{years_a[-1]}"
    else:
        title = f"<{A.exp}> \n {season.upper()} {years_a[0]}-{years_a[-1]}"
    #title = titles[ix]

    n=ix+1
    Axes1 = Pu.axes_def(n=n,nxplo=nxplo,nyplo=nyplo, ypan=0.8 ) 
    ax1 = fig.add_axes( Axes1 )
    co1=ax1.contourf(lat_a,zlev_a,AAa_sz ,levels=mlev, cmap=cmap ,extend='both' )
    co2=ax1.contour(lat_a,zlev_a,AAa_sz ,levels=mlev, colors='black')
    #ax1.set_title(f"Control <{A.exp}> {season.upper()} {years_a[0]}-{years_a[-1]}" )
    ax1.set_title( title ,fontsize=titlesize )
    ax1.set_ylim(0,plotTop)
    ax1.set_xlabel('Latitude', fontsize=16 )
    ax1.set_ylabel('Z (km)', fontsize=16 )
    cb=plt.colorbar(co1)
    ax1.text(-0.08, 1.05, f"{chr(97 +n-1)})", transform=ax1.transAxes,
        fontsize=20, fontweight='bold', va='top')
    
    # Sponge layer hatching
    ax1.contourf(lat_a, zlev_a, sponge_layer, levels=[0.5, 1.5], hatches=['//'], alpha=0)

    ax1.set_xlim( latlims )

    if ( 'jaxes' in A) and (ix>=0):
        #ax1.plot( A.jax_mean , zlev_a , linewidth=2.5, color='black')
        ax1.plot( A.jax_min , zlev_a , linewidth=2.5, color='black' , linestyle='--')
        ax1.plot( A.jax_max , zlev_a , linewidth=2.5, color='black' , linestyle='--')
        #ax1.plot( A.jax_mean+2*A.jax_std , zlev_a , linewidth=2.5, color='black' , linestyle='--')
        #ax1.plot( A.jax_mean-2*A.jax_std , zlev_a , linewidth=2.5, color='black' , linestyle='--')
        if ( 'jaxes' in Aval):
            #ax1.plot( Aval['jax_mean'] , zlev_v , linewidth=2.5, color='black')
            ax1.plot( Aval['jax_min'] , zlev_v , linewidth=4.5, color='cyan' , linestyle=':')
            ax1.plot( Aval['jax_max'] , zlev_v , linewidth=4.5, color='cyan' , linestyle=':')
            #ax1.plot( Aval['jax_mean']+2*Aval['jax_std'] , zlev_v , linewidth=4.5, color='cyan' , linestyle=':')
            #ax1.plot( Aval['jax_mean']-2*Aval['jax_std'] , zlev_v , linewidth=4.5, color='cyan' , linestyle=':')

    
    ix=ix+1

if (Aval['data_exists']==True):
    for ix in np.arange( start=numx,stop=numx+1 ):

        #lon_v = Al[0]['lon_v']
        #lat_v = Al[0]['lat_v']
        #zlev_v = Al[0]['zlev_v']
        #AAv_sz = Al[0]['AAv_sz']
        lon_v = Aval['lon']
        lat_v = Aval['lat']
        zlev_v = Aval['zlev']
        AAv_sz = scale*Aval['AA_sz']

        
        if (use_pmid_for_z==True):
            Zlev_v = zlev_v
            # Create 2D arrays
            lat_v, Lev_v = np.meshgrid(lat_v, Aval['lev'] )
        else:
            # Create 2D arrays
            Lat_v, Zlev_v = np.meshgrid(lat_v, zlev_v)

        print( np.shape( lat_v ) )
        print( np.shape( zlev_v ) )
        
        
        cmap='gist_ncar'
        title = f"<{valData}> {season.upper()} {years_Uv} " 
        n=ix+1
        Axes1 = Pu.axes_def(n=n,nxplo=nxplo,nyplo=nyplo, ypan=0.8 ) 
        ax1 = fig.add_axes( Axes1 )
        co1=ax1.contourf(lat_v,zlev_v,AAv_sz ,levels=ulev, cmap=cmap , extend='both' )
        co2=ax1.contour(lat_v,zlev_v,AAv_sz ,levels=ulev, colors='black')
        ax1.set_title( title ,fontsize=titlesize )
        ax1.set_ylim(0,plotTop)
        ax1.set_xlabel('Latitude', fontsize=16 )
        ax1.set_ylabel('Z (km)', fontsize=16 )
        cb=plt.colorbar(co1)
        ax1.text(-0.08, 1.05, f"{chr(97 +n-1)})", transform=ax1.transAxes,
            fontsize=20, fontweight='bold', va='top')
        ax1.set_xlim( latlims )
        
        if ( 'jaxes' in Aval):
            #ax1.plot( Aval['jax_mean'] , zlev_v , linewidth=2.5, color='black')
            ax1.plot( Aval['jax_min'] , zlev_v , linewidth=2.5, color='black' , linestyle='--')
            ax1.plot( Aval['jax_max'] , zlev_v , linewidth=2.5, color='black' , linestyle='--')
            #ax1.plot( Aval['jax_mean']+2*Aval['jax_std'] , zlev_v , linewidth=2.5, color='black' , linestyle='--')
            #ax1.plot( Aval['jax_mean']-2*Aval['jax_std'] , zlev_v , linewidth=2.5, color='black' , linestyle='--')


FigName = f"Fig_zonal_{fld}.png"

print( FigName) 
Pu.save_link(png_filename = FigName, dpi=600 )

n_next_startlabel = n


In [ ]:
print( scale )

A.exp

In [ ]:
kbot=9
for k in np.arange( kbot ):
    print( f' ----- {100*Al[0].X.ilev[k].values :6.2f} ') 
    print( f'  ***    {100*Al[0].X.lev[k].values :6.2f} ') 

In [ ]:
# Get information for Common grid ... here = fv1x1 x L93

#CommonGridInfo = GrU.gridInfo(grid='fv1x1', Vgrid='L93' )
CommonGridInfo = GrU.gridInfo(grid='fv1x1', Vgrid='ERA5' )
lat_comm,lon_comm   = GrU.latlon( scrip= CommonGridInfo['scrip'], Hkey=CommonGridInfo['Hkey'] )
plev_comm,zlev_comm = GrU.vgrid( VgridFile= CommonGridInfo['VgridFile'] )
Lat_comm , Zlev_comm = np.meshgrid(lat_comm, zlev_comm)
CommonGrid ={ 'lat': Lat_comm , 'zlev':Zlev_comm }

if (len(np.shape(lat_v))==1) and (len(np.shape(zlev_v))==1):
    Lat_v , Zlev_v = np.meshgrid(lat_v, zlev_v) #zlev_v[:,0])
if (len(np.shape(lat_v))==1) and (len(np.shape(zlev_v))==2):
    Lat_v , Zlev_v = np.meshgrid(lat_v, zlev_v[:,0])
elif (len(np.shape(lat_v))==2) and (len(np.shape(zlev_v))==2):
    Lat_v , Zlev_v = lat_v, zlev_v

print( np.shape( zlev_v) )
print( np.shape( lat_v) )


print( np.shape( AAv_sz) )
print( np.shape( Zlev_v) )
print( np.shape( Lat_v) )

AAv_sz_comm = nuti.ZsY_x_ZtY( y_src=Zlev_v , x_src=Lat_v , data_src= AAv_sz , y_tgt=CommonGrid['zlev'], x_tgt=CommonGrid['lat'] )

Aval['AA_sz_comm'] = AAv_sz_comm 

#plt.contour( Lat_comm, Zlev_comm , AAv_sz_comm  - Al[1].AA_sz )

#print( zlev_comm )

In [ ]:
print( np.shape(Lat_comm ))
print( np.shape(Zlev_comm ))
print( np.shape(Lat_v ))
print( np.shape(Zlev_v ))


In [ ]:
print( lat_comm.shape )
print( lat_v.shape )


In [ ]:
#nxplo,nyplo=4,1
#fig = plt.figure(figsize=(24, 5))

numx=len( Al )
plot_bias_vs_obs = False #True
if (plot_bias_vs_obs == True):
    nxplo,nyplo=numx+1,1
else:
    nxplo,nyplo=numx+1,1

latlims = (-90.,90. )
#latlims = (-90.,0. )
plotTop=85 #85 #40 # 15 #10 #85


fig = plt.figure(figsize=(8*nxplo, 9))
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16

ulev=np.linspace( -60,140,num=21) 
ulevx = np.append(ulev, [160,180,200])

ulev=np.linspace( -0.05,0.05,num=41) 
ulevx=ulev


####################################
# Use a centarlized functiin to define 
# intervals 
####################################
#def contour_intervals( fld, **kwargs ): return flev,dlev,scale

#dlev=np.linspace( -20,20,num=21) 

ulev,dlev,scale = puti.contour_intervals( fld, zonal_mean=1 )
#ulev=np.linspace(-450,450,num=51)
#ulev=np.linspace(-5,5,num=51)
ulevx=ulev
print(dlev)
cmap='bwr_r' #'gist_ncar'
cmap='gist_ncar'
titlesize=14 #24 #18

BoxParm =['alpha_gw_movmtn', 'effgw_rdg_resid','use_gw_rdg_resid']
BoxParm =['use_gw_convect_dp','use_gw_movmtn_pbl','use_gw_front','use_gw_rdg_resid','use_gw_rdg_beta','use_gw_oro']
BoxParm =['use_gw_movmtn_pbl','use_gw_rdg_resid','movmtn_source','alpha_gw_movmtn', 'effgw_movmtn_pbl','effgw_rdg_resid',]
BoxParm =['movmtn_source','alpha_gw_movmtn', 'effgw_movmtn_pbl','taubgnd','effgw_cm','effgw_rdg_beta','effgw_rdg_resid',]
#BoxParm =['use_gw_movmtn_pbl','movmtn_source','alpha_gw_movmtn', 'effgw_movmtn_pbl',]  #'effgw_rdg_resid',]


titles=[ r'1$^\circ$ control','14km simulation',r'1$^\circ$ w/ additional GW source','yy',]
titles=[ r'1$^\circ$ control',r'1$^\circ$ w/out GW parameterization','yy',]
ix=0

for A in Al:
    
    AA_sz = A['AA_sz'] 
    lat_a,zlev_a = A['latx'], A['zlev'] 
    print(np.shape(lat_a))
    print(np.shape(zlev_a))
    if (len(np.shape(lat_a))==1) and (len(np.shape(zlev_a))==1):
        Lat_a , Zlev_a = np.meshgrid(lat_a,zlev_a) #zlev_v[:,0])
    if (len(np.shape(lat_a))==1) and (len(np.shape(zlev_a))==2):
        Lat_a , Zlev_a = np.meshgrid(lat_a,zlev_a[:,0]) #zlev_v[:,0])
    elif (len(np.shape(lat_a))==2) and (len(np.shape(zlev_a))==2):
        Lat_a , Zlev_a =lat_a,zlev_a

    print(np.shape(Lat_a))
    AA_sz_comm = nuti.ZsY_x_ZtY( y_src=Zlev_a , x_src=Lat_a , data_src= AA_sz , y_tgt=CommonGrid['zlev'], x_tgt=CommonGrid['lat'] )
    A['AA_sz_comm'] = AA_sz_comm 
    AA_sz_comm = AA_sz_comm - Aval['AA_sz_comm']

    zlev_comm = CommonGrid['zlev']
    lat_comm  = CommonGrid['lat'] 
    mlev=dlev *2 #* 2
    cmap='bwr' #_r'
    #plotTop=85 #40 # 15 #10 #85
        
    years_a = A['years']
    if 'gw_parameters' in A:
        parameters=A['gw_parameters']
    else:
        parameters=None

    if ('desc' in A) and (use_desc_in_title == True):
        title = f"<{A.exp}> \n {A.desc} \n {season.upper()} {years_a[0]}-{years_a[-1]}"
    else:
        title = f"<{A.exp}> \n {season.upper()} {years_a[0]}-{years_a[-1]}"
    #title = titles[ix]

    n=ix+1 + n_next_startlabel
    Axes1 = Pu.axes_def(n=n,nxplo=nxplo,nyplo=nyplo, ypan=0.8 ) 
    ax1 = fig.add_axes( Axes1 )

    
    co1=ax1.contourf( lat_comm, zlev_comm, AA_sz_comm ,levels=mlev, cmap=cmap ,extend='both' )
    co2=ax1.contour(  lat_comm, zlev_comm, AA_sz_comm ,levels=mlev, colors='black')

    
    ax1.set_title( title ,fontsize=titlesize )
    ax1.set_ylim(0,plotTop)
    ax1.set_xlabel('Latitude', fontsize=16 )
    ax1.set_ylabel('Z (km)', fontsize=16 )
    cb=plt.colorbar(co1)
    ax1.text(-0.08, 1.05, f"{chr(97 +n-1)})", transform=ax1.transAxes,
        fontsize=20, fontweight='bold', va='top')
    
    # Sponge layer hatching
    #ax1.contourf(lat_comm, zlev_a, sponge_layer, levels=[0.5, 1.5], hatches=['//'], alpha=0)

    ax1.set_xlim( latlims )

    ix=ix+1

if (Aval['data_exists']==True):
    for ix in np.arange( start=numx,stop=numx+1 ):

        if (regrid_val_x_model == True) or (all_x_fv1x1_zERA==True):
            lon_v = Al[0]['lon_v']
            lat_v = Al[0]['lat_v']
            zlev_v = Al[0]['zlev_v']
            AAv_sz = Al[0]['AAv_sz']
            
        cmap='gist_ncar'
        title = f"<{valData}> {season.upper()} {years_Uv} " 
        n=ix+1+n_next_startlabel
        Axes1 = Pu.axes_def(n=n,nxplo=nxplo,nyplo=nyplo, ypan=0.8 ) 
        ax1 = fig.add_axes( Axes1 )
        co1=ax1.contourf(lat_v,zlev_v,AAv_sz ,levels=ulev, cmap=cmap , extend='both' )
        co2=ax1.contour(lat_v,zlev_v,AAv_sz ,levels=ulev, colors='black')
        ax1.set_title( title ,fontsize=titlesize )
        ax1.set_ylim(0,plotTop)
        ax1.set_xlabel('Latitude', fontsize=16 )
        ax1.set_ylabel('Z (km)', fontsize=16 )
        cb=plt.colorbar(co1)
        ax1.text(-0.08, 1.05, f"{chr(97 +n-1)})", transform=ax1.transAxes,
            fontsize=20, fontweight='bold', va='top')
        ax1.set_xlim( latlims )

FigName = f"AGU_2024_Cecile_poster_zonalUT.png" #f"{A.exp}_{season.upper()}_{years_a[0]}-{years_a[-1]}_Dst{Dst}_zUT_climo.png"
FigName = f"multiFig.png" #f"{A.exp}_{season.upper()}_{years_a[0]}-{years_a[-1]}_Dst{Dst}_zUT_climo.png"
FigName = f"Figure_1.png" #f"{A.exp}_{season.upper()}_{years_a[0]}-{years_a[-1]}_Dst{Dst}_zUT_climo.png"
FigName = f"Fig_KeyResult4paper_{fld}_biases.png"

print( FigName) 
Pu.save_link(png_filename = FigName, dpi=600 )

In [ ]:
nz,ny = np.shape(Aval['AA_sz_comm'] )

In [ ]:
latSouth , latNorth = -90 , -10
yy= [    np.argmin( np.abs( lat_v - latSouth ) ) , np.argmin( np.abs( lat_v - latNorth) )   ]
for A in Al:
    rms=np.zeros( nz )
    for z in np.arange( nz ):
        latv=lat_comm[z,yy[0]:yy[1]]
        AA=A['AA_sz_comm'][z,yy[0]:yy[1]]
        BB=Aval['AA_sz_comm'][z,yy[0]:yy[1]]
        diff=AA-BB
        wgt = np.cos(np.deg2rad(latv))
        rms[z]=np.sqrt(np.sum((diff**2) * wgt) / np.sum(wgt))
        A['rms_era']=rms


In [ ]:
print( zlev_comm[0:5,0] )

In [ ]:

if fld == 'U':
    unit = "m s$^{-1}$"
    range=[0,40]
elif fld == 'T':
    unit="K"
    range=[0,30]

plt.figure(figsize=(12,12))
for A in Al:
    plt.plot( A.rms_era , zlev_comm[:,0], label=A.exp)
    print( f'{np.average( A.rms_era[0:4] )} {A.exp}')
plt.xlabel(f"RMS error w/ resp ERA5 ({unit})",fontsize=18 )
plt.ylabel("Z (km)",fontsize=18 )
plt.xlim( range )
plt.legend()

FigName = f"Bias_profiles_{fld}_B.png"

print( FigName) 
Pu.save_link(png_filename = FigName, dpi=600 )

In [ ]:
plt.plot( Zlev_a[:,0])
plt.plot( Zlev_a[:,100])
plt.plot( zlev_v )
print( np.shape(CommonGrid['zlev']))


In [ ]:
plt.plot(Aval['AA_sz_comm'][:,0],zlev_comm[:,0],'-o')
plt.plot(Al[0]['AA_sz_comm'][:,0],zlev_comm[:,0],'-x')
plt.plot(Aval['AA_sz'][:,0],zlev_v,'-+')
print( Aval['AA_sz_comm'][0:5,0])

In [ ]:
print( np.shape( Al[1].X.T ))

plt.plot( ( Al[0].X.T[0,:,:]-Al[1].X.T[0,:,:] ).values.flatten() )

In [ ]:
A=Al[0]
plt.scatter( A.X.lat.values, A.X.SOLIN.values[2,:].flatten())